# Video Generation

> **Placeholder.** This notebook will build text-to-video generation on a real open video
> diffusion transformer. The outline below fixes the scope, the theory and the findings; code
> comes later.

---

## Why this topic

Video generation is image generation with a time axis, but the time axis changes everything:
sequence length explodes (seconds of 720p video are hundreds of thousands of latent tokens),
motion must stay consistent, and 2025's frontier models also generate synchronised **audio**.
The field converged on one recipe: a **causal 3D VAE** compresses space and time, a
**video diffusion transformer** trained with **flow matching** denoises the latent, and
distillation plus autoregressive variants push towards real-time and long videos.

## Theory

**Latent compression.** A causal 3D VAE maps a video $x \in \mathbb{R}^{T \times H \times W \times 3}$
to $z \in \mathbb{R}^{T/4 \times H/8 \times W/8 \times C}$ (typical factors); causal in time so the
first frame is encoded alone and images and videos share one model.

**Tokens and attention.** $z$ is patchified into spacetime patches (e.g. $1 \times 2 \times 2$),
giving $N = \tfrac{T}{4}\cdot\tfrac{H}{16}\cdot\tfrac{W}{16}$ tokens. Full 3D attention costs
$O(N^2)$; models use 3D RoPE over $(t, h, w)$ and either full attention (quality) or
factorised / windowed / sparse attention (cost).

**Objective.** Flow matching on the linear path $z_t = (1-t)\,z_0 + t\,\epsilon$ with target
velocity $v = \epsilon - z_0$:
$\mathcal{L} = \mathbb{E}\,\lVert v_\theta(z_t, t, c) - (\epsilon - z_0) \rVert^2$, conditioned on text
$c$ via cross-attention or joint (MM-DiT) attention, sampled with classifier-free guidance.

**Long and real-time video.** Bidirectional denoising of the whole clip limits length. Causal /
autoregressive video diffusion denoises chunk by chunk with a KV cache over past frames;
*self-forcing* trains on the model's own rollouts to remove exposure bias; few-step distillation
gives interactive frame rates. These are the same mechanics behind interactive world models.

**Joint audio–video.** Veo 3 and Sora 2 generate speech, effects and music in sync; open models
add an audio latent stream sharing the transformer.

## Findings from the past year

| Model | Date | What is new |
|---|---|---|
| **Wan 2.2** (Alibaba) | 2025 | Open (Apache-2.0) video DiT with a two-expert **MoE** split by noise level (high-noise layout expert, low-noise detail expert); 5B variant runs on consumer GPUs |
| **Wan 2.5 / 2.6** | 2025–26 | Audio-visual generation; check which weights are open |
| **Veo 3 / 3.1** (Google) | 2025 | Native synchronised audio; closed (Vertex AI / Gemini API) |
| **Sora 2** (OpenAI) | Sep 2025 | DiT over spacetime patches, 10–25 s clips with audio |
| **HunyuanVideo 1.5**, **LTX-2** | 2025–26 | Open; LTX-2 adds native audio; HunyuanVideo 1.5 fits a single 24 GB GPU |

## Planned chapters

| Chapter | What it will build |
|---|---|
| 1) Introduction | From image DiT to video DiT; the sequence-length problem |
| 2) Setup | `Wan-AI/Wan2.2-TI2V-5B`-class checkpoint (or a smaller open video DiT), short prompt set, `DEVICE` |
| 3) The math | Causal 3D VAE factors, spacetime patch count, 3D RoPE, flow-matching loss, CFG |
| 4) From scratch | Token-count calculator; 3D RoPE frequencies; flow-matching Euler sampler driving the real DiT; chunked causal sampling with a frame cache |
| 5) End-to-end | Generate a short clip; measure latent tokens, step count vs quality, temporal consistency (frame-to-frame feature similarity) |
| 6) Variants | Steps vs guidance; MoE expert switch point; full vs windowed attention cost |
| 7) Practical notes | VRAM, offloading, resolution/length trade-offs, evaluation (VBench) |
| 8) Conclusion | Decision table |

## Key references

- Wan Team, *Wan: Open and Advanced Large-Scale Video Generative Models* (2025)
- OpenAI, *Video generation models as world simulators* (Sora, 2024) and Sora 2 system card (2025)
- Google DeepMind, Veo 3 tech report (2025)
- Kong et al., *HunyuanVideo* (2024); Lightricks, *LTX-Video* (2024)
- Huang et al., *Self Forcing: Bridging the Train-Test Gap in Autoregressive Video Diffusion* (2025)
- Huang et al., *VBench* (2024)